# GridStress — SMARD data and temporal structure

A compact, human-readable EDA notebook built directly from the SMARD API record.

The notebook downloads the observed hourly series, inspects and cleans the table, then moves through the figures in a simple sequence: **question → code → figure → interpretation**. The available record spans **2019 through September 2026**, with **2026 explicitly partial**.


## 1. What data will we pull from SMARD?

We use the four physical inputs needed for the residual-load identity plus SMARD's published residual-load series as a cross-check. The requested range is **2019-01-01 to 2026-09-10** in Europe/Berlin time.

In [ ]:
from datetime import datetime, timezone
from pathlib import Path
from zoneinfo import ZoneInfo

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
import requests
from statsmodels.tsa.stattools import acf, pacf

LOCAL_TZ = ZoneInfo("Europe/Berlin")
BASE = "https://www.smard.de/app"
REGION = "DE"
RESOLUTION = "hour"

START = datetime(2019, 1, 1, tzinfo=LOCAL_TZ)   # inclusive
END   = datetime(2026, 9, 10, tzinfo=LOCAL_TZ)  # exclusive

FILTERS = {
    1225: "Wind Offshore",
    4067: "Wind Onshore",
    4068: "Solar",
    410:  "Grid Load",
    4359: "Residual Load (SMARD)",
}

CHECK_ENDPOINTS = True
REFRESH = False

cwd = Path.cwd().resolve()
ROOT = cwd.parent if cwd.name == "notebooks" else cwd
DATA_DIR = ROOT / "data"
DATA_DIR.mkdir(parents=True, exist_ok=True)
CSV_FILE = DATA_DIR / "smard_hourly_2019_2026-09-10.csv"

COLORS = {
    "navy":   "#17365D",
    "blue":   "#2C6EBA",
    "teal":   "#4AA3A5",
    "green":  "#2F8F5B",
    "mint":   "#6FBF9B",
    "gold":   "#D9A53A",
    "orange": "#E95D0F",
    "coral":  "#E76F51",
    "red":    "#B10F0F",
    "purple": "#9274B8",
    "gray":   "#AEB8C5",
    "ink":    "#17365D",
    "muted":  "#707B8C",
    "grid":   "#E3E8EF",
    "cream":  "#F7F3E7",
}

YEAR_COLORS = {
    2019: "#4C78A8", 2020: "#4AA3A5", 2021: "#72B7A1", 2022: "#D9A53A",
    2023: "#E76F51", 2024: "#9274B8", 2025: "#17365D", 2026: "#2F8F5B",
}

plt.rcParams.update({
    "figure.facecolor": "white",
    "axes.facecolor": "white",
    "axes.edgecolor": "#D7DEE8",
    "axes.labelcolor": COLORS["ink"],
    "xtick.color": COLORS["muted"],
    "ytick.color": COLORS["muted"],
    "text.color": COLORS["ink"],
    "font.size": 11,
    "axes.titlesize": 16,
    "axes.titleweight": "bold",
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.titlepad": 12,
    "axes.linewidth": 0.8,
    "legend.frameon": False,
})

LEVEL_CMAP = LinearSegmentedColormap.from_list(
    "gridstress_level",
    [COLORS["green"], "#B9DCC7", COLORS["cream"], "#FDBA67", COLORS["orange"], COLORS["red"]],
)
DIV_CMAP = LinearSegmentedColormap.from_list(
    "gridstress_diverging",
    [COLORS["teal"], "#DCEEEF", "#FFFFFF", "#F8D1C5", COLORS["coral"]],
)
SUPPORT_CMAP = LinearSegmentedColormap.from_list(
    "gridstress_support",
    ["#F7F9FC", "#D7E6F3", "#82B6D9", COLORS["blue"], COLORS["navy"]],
)
RAMP_CMAP = LinearSegmentedColormap.from_list(
    "gridstress_ramp",
    ["#EDF6F3", "#A9D7CC", "#F5D08A", "#F28E2B", "#D95F0E", COLORS["red"]],
)
WIND_SOLAR_CMAP = LinearSegmentedColormap.from_list(
    "gridstress_wind_solar",
    ["#F8F3E7", "#F3C879", "#9BD0C3", COLORS["teal"], COLORS["green"]],
)

print(f"{len(FILTERS)} SMARD series | {RESOLUTION} | {START:%Y-%m-%d} to {END:%Y-%m-%d}")
print("Cache:", CSV_FILE)

## 2. How does the API import work?

SMARD serves weekly packages rather than an arbitrary free-form date range. We first read the package index, download only packages that overlap our requested window, and trim the values to the exact start/end interval.

In [ ]:
def to_ms(dt):
    return int(dt.timestamp() * 1000)


def to_utc(t_ms):
    return pd.Timestamp(t_ms, unit="ms", tz="UTC")


def index_url(filter_id, resolution=RESOLUTION, region=REGION):
    return f"{BASE}/chart_data/{filter_id}/{region}/index_{resolution}.json"


def fetch(filter_id, start=START, end=END, resolution=RESOLUTION, region=REGION, session=None):
    """Download one SMARD series over [start, end)."""
    a, b = to_ms(start), to_ms(end)
    http = session or requests

    response = http.get(index_url(filter_id, resolution, region), timeout=30)
    response.raise_for_status()
    packages = sorted(response.json()["timestamps"])

    wanted = [
        p for i, p in enumerate(packages)
        if p < b and (i + 1 == len(packages) or packages[i + 1] > a)
    ]

    values = {}
    for n, package in enumerate(wanted, 1):
        url = (
            f"{BASE}/chart_data/{filter_id}/{region}/"
            f"{filter_id}_{region}_{resolution}_{package}.json"
        )
        r = http.get(url, timeout=30)
        r.raise_for_status()
        for t, value in r.json()["series"]:
            if a <= t < b:
                values[to_utc(t)] = value
        if n % 50 == 0 or n == len(wanted):
            print(f"    {n}/{len(wanted)} packages", end="\r")
    return values


def check_endpoints():
    ok = True
    for filter_id, name in FILTERS.items():
        try:
            r = requests.get(index_url(filter_id), timeout=15)
            status = "ok" if r.status_code == 200 else f"HTTP {r.status_code}"
        except Exception as exc:
            status = type(exc).__name__
        ok &= status == "ok"
        print(f"  {filter_id:>5}  {name:<24} {status}")
    return ok


if CHECK_ENDPOINTS:
    all_ok = check_endpoints()
    print("\nAll endpoints reachable." if all_ok else "\nWarning: at least one endpoint did not respond.")
else:
    print("Endpoint check skipped.")

**Interpretation.** This is intentionally close to the simple API notebook: package index → overlapping weekly packages → exact trimming. The extra care here is mostly around UTC storage, reproducible caching, explicit missing hours and the residual-load cross-check below.

## 3. Did we download a sensible hourly table?

If a cached CSV already exists, we reuse it unless `REFRESH=True`. Otherwise the five series are downloaded directly from SMARD.

In [ ]:
start_utc = pd.Timestamp(START).tz_convert("UTC")
end_utc = pd.Timestamp(END).tz_convert("UTC")
expected_index = pd.date_range(start_utc, end_utc, freq="h", inclusive="left")

if CSV_FILE.exists() and not REFRESH:
    raw = pd.read_csv(CSV_FILE)
    raw["timestamp_utc"] = pd.to_datetime(raw["timestamp_utc"], utc=True)
    df = raw.set_index("timestamp_utc").sort_index()
    print("Loaded cached CSV.")
else:
    data = {}
    with requests.Session() as session:
        for filter_id, name in FILTERS.items():
            data[name] = fetch(filter_id, session=session)
            print(f"{name:<24} {len(data[name]):>7} values")
    df = pd.DataFrame(data).sort_index()
    df.index.name = "timestamp_utc"
    df.reset_index().to_csv(CSV_FILE, index=False)
    print("Saved:", CSV_FILE)

print("\nDownloaded table:", df.shape)

## 4. What did we actually download?

Before plotting anything, look at the table like a person would: where it starts and ends, what the first and last rows contain, what a few observations look like in the middle, and whether any column has obvious missingness or strange ranges.

In [ ]:
print(f"Rows × columns: {df.shape[0]:,} × {df.shape[1]}")
print("First timestamp:", df.index.min())
print("Last timestamp: ", df.index.max())

print("\nFirst observations")
display(df.head(6))

print("\nLast observations")
display(df.tail(6))

print("\nA few observations from the middle")
sample_positions = np.linspace(0, len(df) - 1, 6, dtype=int)
display(df.iloc[sample_positions])

overview = pd.DataFrame({
    "dtype": df.dtypes.astype(str),
    "missing": df.isna().sum(),
    "min": df.min(numeric_only=True),
    "median": df.median(numeric_only=True),
    "max": df.max(numeric_only=True),
})

print("\nQuick column check")
display(overview)


**Interpretation.** This is the working hourly table before the EDA starts. Missing physical hours remain visible as `NaN`—nothing is interpolated—and the simple range check makes it easy to catch a bad series before trusting any figure.

## 5. What cleaning is needed before analysis?

Keep the cleaning deliberately small and visible: coerce the API values to numeric, remove duplicate timestamps if any appear, make missing physical hours explicit, leave gaps as `NaN` rather than interpolating them, and derive residual load from

`grid load − onshore wind − offshore wind − solar`.

SMARD's own residual-load series stays in the table as a cross-check.

In [ ]:
series_cols = list(FILTERS.values())

# Numeric coercion keeps unexpected strings from silently entering the analysis.
for col in series_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")

# Keep one timestamp per physical hour.
duplicate_rows = int(df.index.duplicated(keep="last").sum())
if duplicate_rows:
    df = df[~df.index.duplicated(keep="last")]

# Make absent physical hours explicit. We do not interpolate them.
df = df.sort_index().reindex(expected_index)
df.index.name = "timestamp_utc"

# Primitive physical quantities should not be negative.
physical_inputs = ["Grid Load", "Wind Onshore", "Wind Offshore", "Solar"]
negative_counts = {}
for col in physical_inputs:
    bad = df[col] < 0
    negative_counts[col] = int(bad.sum())
    if bad.any():
        df.loc[bad, col] = np.nan

# Transparent residual-load identity used throughout the EDA.
df["Residual Load"] = (
    df["Grid Load"]
    - df["Wind Onshore"]
    - df["Wind Offshore"]
    - df["Solar"]
)

gap = (df["Residual Load (SMARD)"] - df["Residual Load"]).abs()

print("Duplicate timestamps removed:", duplicate_rows)
print("Negative primitive values set to NaN:", negative_counts)
print("\nMissing values after alignment:")
print(df[series_cols].isna().sum())
print(f"\nMedian |SMARD residual − identity|: {gap.median():,.2f}")
print(f"P95    |SMARD residual − identity|: {gap.quantile(.95):,.2f}")


**Interpretation.** The cleaning step does not smooth or fill the time series. It standardizes the table, exposes gaps, checks the primitive physical series, and creates the transparent residual-load identity used below.

## 6. Quick visual check — do the downloaded series look physically plausible?

Before summarizing several years, look at one ordinary seven-day window. Load should retain a daily rhythm, solar should switch off at night, and residual load should fall when wind/solar support is stronger.

In [ ]:

valid_core = df[["Grid Load", "Wind Onshore", "Wind Offshore", "Solar", "Residual Load"]].notna().all(axis=1)
valid_positions = np.flatnonzero(valid_core.to_numpy())
splits = np.split(valid_positions, np.where(np.diff(valid_positions) > 1)[0] + 1)
longest = max((s for s in splits if len(s)), key=len)
seven_days = df.iloc[longest[:24*7]].copy() / 1000.0
x_local = seven_days.index.tz_convert(LOCAL_TZ)

fig, ax = plt.subplots(figsize=(13.5, 5.9))

ax.plot(x_local, seven_days["Grid Load"], lw=2.7, color=COLORS["navy"], label="Grid load")
ax.plot(x_local, seven_days["Residual Load"], lw=2.6, color=COLORS["orange"], label="Residual load")
ax.plot(x_local, seven_days["Wind Onshore"], lw=1.9, color=COLORS["teal"], label="Onshore wind")
ax.plot(x_local, seven_days["Wind Offshore"], lw=1.9, color=COLORS["blue"], label="Offshore wind")
ax.plot(x_local, seven_days["Solar"], lw=1.9, color=COLORS["gold"], label="Solar")

fig.suptitle(
    "Do the downloaded series look physically plausible?",
    x=0.055, y=0.97, ha="left",
    fontsize=18, fontweight="bold",
)
handles, labels = ax.get_legend_handles_labels()
fig.legend(
    handles, labels,
    ncol=3, frameon=False,
    loc="upper center",
    bbox_to_anchor=(0.60, 0.925),
)

ax.set_ylabel("Power (GW)")
ax.set_xlabel("Europe/Berlin time")
ax.grid(axis="y", color=COLORS["grid"], lw=0.8)
plt.subplots_adjust(left=0.07, right=0.98, bottom=0.12, top=0.80)
plt.show()


**Interpretation.** This is a simple sanity check rather than an analytical result. The daily load cycle, daytime solar pulse and residual-load response should all be recognizable before we move on to aggregated summaries.

## 7. What data do we actually have?

The first summary is coverage by month. A value of 1 means all physical hours in that calendar month are jointly valid across load, wind and solar; partial 2026 months remain visibly partial rather than being silently treated as complete.

In [ ]:
local_index = df.index.tz_convert(LOCAL_TZ)
analysis = pd.DataFrame(index=df.index)
analysis["local_time"] = local_index
analysis["year"] = local_index.year
analysis["month"] = local_index.month
analysis["day"] = local_index.day
analysis["hour"] = local_index.hour
analysis["weekday"] = local_index.dayofweek
analysis["date_local"] = local_index.date
analysis["season"] = pd.Categorical(
    np.select(
        [analysis["month"].isin([12,1,2]), analysis["month"].isin([3,4,5]), analysis["month"].isin([6,7,8])],
        ["Winter", "Spring", "Summer"],
        default="Autumn",
    ),
    categories=["Winter", "Spring", "Summer", "Autumn"],
    ordered=True,
)

for col in ["Grid Load", "Wind Onshore", "Wind Offshore", "Solar", "Residual Load"]:
    analysis[col] = df[col] / 1000.0
analysis["Wind + Solar"] = analysis["Wind Onshore"] + analysis["Wind Offshore"] + analysis["Solar"]
analysis["Wind+solar / load"] = analysis["Wind + Solar"] / analysis["Grid Load"]
analysis["|1h residual ramp|"] = analysis["Residual Load"].diff().abs()
analysis["core_valid"] = valid_core.to_numpy()

# Full calendar-month denominators preserve DST and show partial-window months as partial.
coverage = pd.DataFrame(index=range(2019, 2027), columns=range(1, 13), dtype=float)
for year in coverage.index:
    for month in coverage.columns:
        month_start = pd.Timestamp(year=year, month=month, day=1, tz=LOCAL_TZ)
        next_month = month_start + pd.DateOffset(months=1)
        month_utc = pd.date_range(month_start.tz_convert("UTC"), next_month.tz_convert("UTC"), freq="h", inclusive="left")
        in_research = month_utc[(month_utc >= start_utc) & (month_utc < end_utc)]
        if len(in_research) == 0:
            coverage.loc[year, month] = np.nan
            continue
        observed = analysis.loc[analysis.index.intersection(in_research), "core_valid"].sum()
        coverage.loc[year, month] = observed / len(month_utc)

fig, ax = plt.subplots(figsize=(13, 5.8))
im = ax.imshow(coverage.values, aspect="auto", vmin=0, vmax=1, cmap=SUPPORT_CMAP)
ax.set_title("How complete is the observed record?", loc="left")
ax.set_xticks(range(12), ["Jan","Feb","Mar","Apr","May","Jun","Jul","Aug","Sep","Oct","Nov","Dec"])
ax.set_yticks(range(8), [str(y) if y < 2026 else "2026 · partial" for y in coverage.index])
ax.set_xlabel("Calendar month")
ax.set_ylabel("")
cb = fig.colorbar(im, ax=ax, fraction=0.025, pad=0.02)
cb.set_label("Joint-valid fraction of full calendar month")
plt.tight_layout()
plt.show()

**Interpretation.** The monthly support map separates complete months from genuinely partial ones. The 2026 record stops inside September, so it remains explicitly partial.

## 8. What does the research record look like in one glance?

The left panel shows the overall residual-load distribution. The right panel shows how quickly the upper tail of absolute one-hour residual-load changes grows toward the extreme percentiles.

In [ ]:
residual = analysis.loc[analysis["core_valid"], "Residual Load"].dropna()
ramps = analysis.loc[analysis["core_valid"], "|1h residual ramp|"].dropna()
qs = [0.80, 0.90, 0.95, 0.975, 0.99, 0.995]
ramp_q = ramps.quantile(qs)

fig, axes = plt.subplots(
    1, 2,
    figsize=(14.5, 5.8),
    gridspec_kw={"width_ratios": [1.15, 1]},
    constrained_layout=True,
)

# Residual-load distribution
axes[0].hist(
    residual,
    bins=55,
    color=COLORS["teal"],
    alpha=0.78,
    edgecolor="white",
    linewidth=0.5,
)
median = residual.median()
axes[0].axvline(median, color=COLORS["navy"], ls="--", lw=2)

y_top = axes[0].get_ylim()[1]
axes[0].annotate(
    f"Median {median:.1f} GW",
    xy=(median, y_top * 0.90),
    xytext=(10, 0),
    textcoords="offset points",
    ha="left",
    va="center",
    color=COLORS["navy"],
    fontsize=10.5,
    bbox=dict(facecolor="white", edgecolor="none", alpha=0.88, pad=1.8),
)

axes[0].set_xlabel("Residual load (GW)")
axes[0].set_ylabel("Observed hours")
axes[0].grid(axis="y", color=COLORS["grid"], lw=0.8)

# Upper tail of one-hour residual ramps
xq = np.array(qs) * 100
axes[1].plot(
    xq,
    ramp_q.values,
    color=COLORS["orange"],
    marker="o",
    markersize=5.5,
    lw=2.5,
    zorder=2,
)

label_offsets = {
    0.90: (10, 10),
    0.95: (10, 10),
    0.99: (-58, 10),
}
for q, offset in label_offsets.items():
    axes[1].annotate(
        f"P{int(q * 100)} {ramp_q.loc[q]:.2f}",
        xy=(q * 100, ramp_q.loc[q]),
        xytext=offset,
        textcoords="offset points",
        ha="left",
        va="bottom",
        color=COLORS["navy"],
        fontsize=10,
        bbox=dict(facecolor="white", edgecolor="none", alpha=0.90, pad=1.6),
        zorder=4,
    )

# Give the annotations deliberate whitespace.
axes[1].margins(x=0.05, y=0.10)
axes[1].set_xlabel("Ramp percentile")
axes[1].set_ylabel("|1-hour residual ramp| (GW/h)")
axes[1].grid(axis="y", color=COLORS["grid"], lw=0.8)

fig.suptitle(
    "What does the observed operating range look like?",
    x=0.055,
    ha="left",
    fontsize=18,
    fontweight="bold",
)
plt.show()

**Interpretation.** Residual load is concentrated around a clear central range, while the one-hour ramp tail bends upward sharply near the highest percentiles. Typical operating level and extreme short-term change are therefore different questions.

## 9. Has the residual-load level shifted by year?

To keep 2026 comparable, each year is restricted to the same calendar span available in 2026. The line is the P10–P90 interval and the dot is the median.

In [ ]:

latest_2026_local = analysis.loc[
    (analysis["year"] == 2026) & analysis["core_valid"], "local_time"
].max()
end_month, end_day = latest_2026_local.month, latest_2026_local.day

matched = analysis[
    (analysis["month"] < end_month)
    | ((analysis["month"] == end_month) & (analysis["day"] <= end_day))
].copy()
matched = matched[matched["core_valid"]]

annual = matched.groupby("year")["Residual Load"].agg(
    median="median",
    p10=lambda s: s.quantile(.10),
    p90=lambda s: s.quantile(.90),
    n="count",
)

fig, ax = plt.subplots(figsize=(11.8, 6.1), constrained_layout=True)

for year, row in annual.iterrows():
    color = YEAR_COLORS[int(year)]
    ax.plot(
        [row["p10"], row["p90"]],
        [year, year],
        lw=5.5,
        color=color,
        alpha=0.23,
        solid_capstyle="round",
        zorder=1,
    )
    ax.scatter(
        row["median"], year,
        s=110,
        color=color,
        edgecolor="white",
        linewidth=0.8,
        zorder=3,
    )
    ax.text(
        row["p90"] + 0.45, year,
        f"n={int(row['n']):,}",
        va="center", ha="left",
        fontsize=9.6,
        color=COLORS["muted"],
    )

ax.set_yticks(
    list(annual.index),
    [str(y) if y < 2026 else "2026 · partial" for y in annual.index],
)
ax.invert_yaxis()
ax.set_xlabel("Residual load (GW)")
ax.set_ylabel("")
ax.set_title("Residual-load level by year", loc="left")
ax.grid(axis="y", color=COLORS["grid"], lw=0.8)
ax.set_xlim(left=min(0, annual["p10"].min() - 2), right=annual["p90"].max() + 7)
plt.show()


**Interpretation.** The annual intervals show whether the centre and spread of residual load move together or separately. Matching the calendar span avoids making the partial 2026 record look directly comparable to full earlier years.

## 10. What is the national daily rhythm?

Average each source by local clock hour to see the basic daily geometry before adding calendar interactions.

In [ ]:

hourly_profile = analysis.loc[analysis["core_valid"]].groupby("hour")[[
    "Grid Load", "Wind Onshore", "Wind Offshore", "Solar", "Residual Load"
]].mean()

fig, ax = plt.subplots(figsize=(13.7, 6.0))

ax.plot(hourly_profile.index, hourly_profile["Grid Load"], lw=2.9, color=COLORS["navy"], label="Grid load")
ax.plot(hourly_profile.index, hourly_profile["Wind Onshore"], lw=2.0, color=COLORS["teal"], label="Onshore wind")
ax.plot(hourly_profile.index, hourly_profile["Wind Offshore"], lw=2.0, color=COLORS["blue"], label="Offshore wind")
ax.plot(hourly_profile.index, hourly_profile["Solar"], lw=2.1, color=COLORS["gold"], label="Solar")
ax.plot(hourly_profile.index, hourly_profile["Residual Load"], lw=2.9, color=COLORS["orange"], label="Residual load")

fig.suptitle(
    "The national daily rhythm",
    x=0.055, y=0.97, ha="left",
    fontsize=18, fontweight="bold",
)
handles, labels = ax.get_legend_handles_labels()
fig.legend(
    handles, labels,
    ncol=3, frameon=False,
    loc="upper center",
    bbox_to_anchor=(0.61, 0.925),
)

ax.set_xlabel("Local clock hour")
ax.set_ylabel("Power (GW)")
ax.set_xticks(range(0,24,3), [f"{h:02d}:00" for h in range(0,24,3)])
ax.grid(axis="y", color=COLORS["grid"], lw=0.8)
plt.subplots_adjust(left=0.07, right=0.98, bottom=0.12, top=0.80)
plt.show()


**Interpretation.** The load profile has morning and evening structure, while solar creates a strong midday reduction in residual load. Wind is flatter on average, so the residual-load curve reflects a different daily shape from demand alone.

## 11. How does the daily shape depend on calendar context?

Instead of three separate figures, month×hour, weekday×hour and season×hour are shown together on one common residual-load colour scale.

In [ ]:
valid = analysis.loc[analysis["core_valid"]].copy()
month_names = ["Jan","Feb","Mar","Apr","May","Jun","Jul","Aug","Sep","Oct","Nov","Dec"]
weekday_names = ["Mon","Tue","Wed","Thu","Fri","Sat","Sun"]
season_names = ["Winter","Spring","Summer","Autumn"]

m = valid.pivot_table(
    index="month", columns="hour", values="Residual Load", aggfunc="mean"
).reindex(range(1, 13))
w = valid.pivot_table(
    index="weekday", columns="hour", values="Residual Load", aggfunc="mean"
).reindex(range(7))
s = valid.pivot_table(
    index="season", columns="hour", values="Residual Load", aggfunc="mean"
).reindex(season_names)

vmin = min(
    np.nanmin(m.values),
    np.nanmin(w.values),
    np.nanmin(s.values),
)
vmax = max(
    np.nanmax(m.values),
    np.nanmax(w.values),
    np.nanmax(s.values),
)

# Explicit fourth column for the colour scale: it can never sit on a heatmap.
fig = plt.figure(figsize=(16.5, 6.4), constrained_layout=True)
gs = fig.add_gridspec(
    1, 4,
    width_ratios=[1.08, 1.08, 1.00, 0.045],
    wspace=0.18,
)

axes = [
    fig.add_subplot(gs[0, 0]),
    fig.add_subplot(gs[0, 1]),
    fig.add_subplot(gs[0, 2]),
]
cax = fig.add_subplot(gs[0, 3])

for ax, table, labels, title in [
    (axes[0], m, month_names, "Month × hour"),
    (axes[1], w, weekday_names, "Weekday × hour"),
    (axes[2], s, season_names, "Season × hour"),
]:
    im = ax.imshow(
        table.values,
        aspect="auto",
        cmap=LEVEL_CMAP,
        vmin=vmin,
        vmax=vmax,
        interpolation="nearest",
    )
    ax.set_title(title, loc="left", fontsize=13, pad=8)
    ax.set_yticks(range(len(labels)), labels)
    ax.set_xticks(range(0, 24, 6), ["00", "06", "12", "18"])
    ax.set_xlabel("Local hour", labelpad=6)

cb = fig.colorbar(im, cax=cax)
cb.set_label("Mean residual load (GW)", labelpad=10)

fig.suptitle(
    "How does residual load vary across calendar context?",
    x=0.055,
    ha="left",
    fontsize=18,
    fontweight="bold",
)
plt.show()

**Interpretation.** The three panels make the hierarchy visible in one place: hour-of-day structure is strong, but its level and midday valley change with month, weekday and season. Using one shared scale keeps those differences directly comparable.

## 12. How much of the 24-hour shape fits into a few harmonics?

Fit one, two and three daily Fourier harmonics to the observed mean 24-hour residual-load profile. This is descriptive reconstruction, not a forecasting model.

In [ ]:
def harmonic_fit(y, n_harmonics):
    h = np.arange(24)
    cols = [np.ones_like(h, dtype=float)]
    for k in range(1, n_harmonics + 1):
        cols.extend([
            np.sin(2*np.pi*k*h/24),
            np.cos(2*np.pi*k*h/24),
        ])
    X = np.column_stack(cols)
    mask = np.isfinite(y)
    beta = np.linalg.lstsq(X[mask], y[mask], rcond=None)[0]
    return X @ beta

mean24 = valid.groupby("hour")["Residual Load"].mean().reindex(range(24)).values
h = np.arange(24)

fig, ax = plt.subplots(figsize=(13, 5.8))
ax.plot(h, mean24, color=COLORS["navy"], marker="o", lw=2.6, label="Observed mean")
ax.plot(h, harmonic_fit(mean24, 1), color=COLORS["teal"], ls="--", lw=2.2, label="1 harmonic")
ax.plot(h, harmonic_fit(mean24, 2), color="#FFB55A", ls="--", lw=2.2, label="2 harmonics")
ax.plot(h, harmonic_fit(mean24, 3), color=COLORS["purple"], ls="--", lw=2.2, label="3 harmonics")
ax.set_title("How much shape fits into a few harmonics?", loc="left")
ax.set_xlabel("Local clock hour")
ax.set_ylabel("Residual load (GW)")
ax.grid(axis="y", color=COLORS["grid"], lw=0.8)
ax.legend(ncol=4, frameon=False, loc="upper center", bbox_to_anchor=(0.5, 1.16))
plt.tight_layout()
plt.show()

**Interpretation.** One harmonic captures only broad daily curvature. Adding the second and third harmonics recovers the morning rise, midday valley and evening peak much more closely, which is why cyclic feature design can go beyond a single sine/cosine pair when the data justify it.

## 13. How much temporal dependence remains in residual load?

Use the longest uninterrupted hourly segment so missing intervals are never squeezed out. The three panels compare total lag association, conditional linear lag structure, and the remaining autocorrelation after removing weekday×hour means.

In [ ]:
# Longest uninterrupted physical hourly segment.
valid_pos = np.flatnonzero(analysis["Residual Load"].notna().to_numpy())
segments = np.split(valid_pos, np.where(np.diff(valid_pos) > 1)[0] + 1)
segment_idx = max((s for s in segments if len(s)), key=len)
segment = analysis.iloc[segment_idx]["Residual Load"].astype(float)

acf_raw = acf(segment.values, nlags=min(336, len(segment)//3), fft=True)
pacf_raw = pacf(
    segment.values,
    nlags=min(48, max(1, len(segment)//4 - 1)),
    method="ywm",
)

seg_frame = pd.DataFrame({"value": segment.values}, index=segment.index)
seg_local = seg_frame.index.tz_convert(LOCAL_TZ)
seg_frame["weekday"] = seg_local.dayofweek
seg_frame["hour"] = seg_local.hour
seg_frame["expected"] = seg_frame.groupby(["weekday", "hour"])["value"].transform("mean")
adjusted = seg_frame["value"] - seg_frame["expected"]
acf_adjusted = acf(adjusted.values, nlags=min(168, len(adjusted)//3), fft=True)

fig, axes = plt.subplots(3, 1, figsize=(13.8, 10.6), constrained_layout=True)

panel_styles = [
    {
        "title": "ACF · total lag association",
        "stem": COLORS["blue"],
        "marker": "#174A7E",
        "band": "#DCE9F5",
    },
    {
        "title": "PACF · conditional linear lag structure",
        "stem": COLORS["orange"],
        "marker": "#A84300",
        "band": "#FBE7D8",
    },
    {
        "title": "ACF after removing weekday × hour means",
        "stem": COLORS["green"],
        "marker": "#1F6B43",
        "band": "#DDEFE5",
    },
]

def stem_panel(ax, values, style, xmax, n_obs, refs=()):
    x = np.arange(len(values))
    conf = 1.96 / np.sqrt(max(n_obs, 1))

    # Each panel owns its own colour family:
    # blue = ACF, orange = PACF, green = adjusted ACF.
    ax.axhspan(-conf, conf, color=style["band"], alpha=0.75, zorder=0)
    ax.vlines(x, 0, values, color=style["stem"], lw=1.25, alpha=0.82, zorder=2)
    ax.scatter(
        x, values,
        s=15,
        color=style["marker"],
        edgecolor="white",
        linewidth=0.35,
        zorder=3,
    )
    ax.axhline(0, color=COLORS["gray"], lw=1.0)

    # Reference lags remain neutral so they never compete with the data.
    for lag, label in refs:
        if lag <= xmax:
            ax.axvline(lag, color="#9AA6B2", ls="--", lw=1.0, alpha=0.8, zorder=1)
            ax.text(
                lag, 1.01, label,
                transform=ax.get_xaxis_transform(),
                ha="center", va="bottom",
                fontsize=9.2,
                color=COLORS["muted"],
                bbox=dict(facecolor="white", edgecolor="none", alpha=0.90, pad=1.0),
            )

    ax.set_title(style["title"], loc="left", fontsize=13)
    ax.set_ylabel("Correlation")
    ax.set_ylim(-1.05, 1.05)
    ax.set_xlim(-1, xmax)
    ax.grid(axis="y", color=COLORS["grid"], lw=0.8)

stem_panel(
    axes[0], acf_raw, panel_styles[0], 336, len(segment),
    refs=[(24, "24 h"), (168, "168 h")],
)
stem_panel(
    axes[1], pacf_raw, panel_styles[1], 48, len(segment),
    refs=[(24, "24 h"), (48, "48 h")],
)
stem_panel(
    axes[2], acf_adjusted, panel_styles[2], 168, len(adjusted),
    refs=[(24, "24 h"), (168, "168 h")],
)

axes[2].set_xlabel("Physical lag (hours)")

fig.suptitle(
    "How much temporal dependence remains in residual load?",
    x=0.055,
    ha="left",
    fontsize=18,
    fontweight="bold",
)
plt.show()

**Interpretation.** Strong daily and weekly recurrence is visible in the ACF, while the PACF separates the shorter direct lag structure. Calendar adjustment reduces some of the obvious pattern but does not remove all dependence, which motivates careful lag/rolling-feature tests later rather than assuming independence.

## 14. Has the daily residual-load shape changed over time?

Compare annual 24-hour mean profiles using the same matched calendar span as 2026.

In [ ]:

annual_shape = matched.groupby(["year", "hour"])["Residual Load"].mean().unstack(0)

preferred_years = [2019, 2022, 2025, 2026]
selected_years = [year for year in preferred_years if year in annual_shape.columns]
baseline_year = selected_years[0]

styles = {
    2019: dict(color=COLORS["blue"],   ls="-",  lw=2.8),
    2022: dict(color=COLORS["orange"], ls="--", lw=2.5),
    2025: dict(color=COLORS["green"],  ls="-.", lw=2.5),
    2026: dict(color=COLORS["purple"], ls=":",  lw=2.8),
}

fig, axes = plt.subplots(
    1, 2,
    figsize=(15.5, 6.2),
    gridspec_kw={"width_ratios": [1.55, 0.85]},
)

# Left: selected yearly profiles.
for year in selected_years:
    st = styles[year]
    axes[0].plot(
        annual_shape.index,
        annual_shape[year],
        color=st["color"],
        ls=st["ls"],
        lw=st["lw"],
        label=str(year) if year < 2026 else "2026 · partial",
    )

axes[0].set_title("Selected annual mean daily profiles", loc="left", fontsize=13)
axes[0].set_xlabel("Local clock hour")
axes[0].set_ylabel("Residual load (GW)")
axes[0].set_xticks(range(0,24,3))
axes[0].grid(axis="y", color=COLORS["grid"], lw=0.8)

# Right: shape difference relative to 2019.
for year in selected_years[1:]:
    st = styles[year]
    diff = annual_shape[year] - annual_shape[baseline_year]
    axes[1].plot(
        annual_shape.index,
        diff,
        color=st["color"],
        ls=st["ls"],
        lw=st["lw"],
        label=str(year) if year < 2026 else "2026 · partial",
    )

axes[1].axhline(0, color="#9AA6B2", ls="--", lw=1.1)
axes[1].set_title(f"Difference vs {baseline_year}", loc="left", fontsize=13)
axes[1].set_xlabel("Local clock hour")
axes[1].set_ylabel("Difference (GW)")
axes[1].set_xticks(range(0,24,3))
axes[1].grid(axis="y", color=COLORS["grid"], lw=0.8)

fig.suptitle(
    "Has the daily residual-load shape changed?",
    x=0.055, y=0.97, ha="left",
    fontsize=18, fontweight="bold",
)

handles, labels = axes[0].get_legend_handles_labels()
fig.legend(
    handles, labels,
    ncol=len(selected_years),
    frameon=False,
    loc="upper center",
    bbox_to_anchor=(0.52, 0.925),
)

plt.subplots_adjust(left=0.07, right=0.98, bottom=0.12, top=0.80, wspace=0.25)
plt.show()


**Interpretation.** The selected anchor years show whether the 24-hour shape is changing rather than simply moving up or down. The difference panel makes the midday valley and evening structure easier to compare without asking eight overlapping lines to carry the whole story.

## 15. Has short-term ramping changed differently from level?

Track the annual P50, P90, P95 and P99 of absolute one-hour residual-load changes on the matched calendar span.

In [ ]:
ramp_year = matched.dropna(subset=["|1h residual ramp|"]).groupby("year")["|1h residual ramp|"].quantile([.50,.90,.95,.99]).unstack()
ramp_year.columns = ["P50","P90","P95","P99"]

fig, ax = plt.subplots(figsize=(13, 5.8))
styles = {
    "P50": (COLORS["gray"], "o"),
    "P90": (COLORS["teal"], "o"),
    "P95": (COLORS["orange"], "D"),
    "P99": (COLORS["red"], "s"),
}
for col, (color, marker) in styles.items():
    ax.plot(ramp_year.index, ramp_year[col], color=color, marker=marker, lw=2.2, label=col)
ax.set_title("Change has a different geometry from level", loc="left")
ax.set_xlabel("Year")
ax.set_ylabel("|Residual ramp| (GW/h)")
ax.grid(axis="y", color=COLORS["grid"], lw=.8)
ax.legend(ncol=4, frameon=False, loc="upper left")
plt.tight_layout()
plt.show()

**Interpretation.** The typical ramp and the extreme tail do not have to move together. The upper percentiles are the more relevant part of the distribution when asking how quickly the system can change over one hour.

## 16. When are the strongest ramps concentrated?

Use the P95 absolute one-hour residual ramp for each season×hour combination.

In [ ]:
ramp_fp = valid.pivot_table(index="season", columns="hour", values="|1h residual ramp|", aggfunc=lambda s: s.quantile(.95)).reindex(season_names)

fig, ax = plt.subplots(figsize=(13, 4.9))
im = ax.imshow(ramp_fp.values, aspect="auto", cmap=RAMP_CMAP)
ax.set_title("When are the strongest one-hour ramps concentrated?", loc="left")
ax.set_yticks(range(4), season_names)
ax.set_xticks(range(0,24,3), [f"{h:02d}" for h in range(0,24,3)])
ax.set_xlabel("Local clock hour")
cb = fig.colorbar(im, ax=ax, fraction=.028, pad=.02)
cb.set_label("P95 |1h residual ramp| (GW/h)")
plt.tight_layout()
plt.show()

**Interpretation.** High-ramp hours cluster differently by season. The map makes it clear that ramp risk is not simply highest whenever residual load itself is highest.

## 17. How does wind-and-solar support move through the calendar?

Average wind+solar divided by load for each local calendar day and lay the years on a common day-of-year axis. Blank cells mean no observed support, not zero generation.

In [ ]:
daily = valid.groupby(["year", "date_local"]).agg(
    load=("Grid Load", "mean"),
    wind_solar=("Wind + Solar", "mean"),
).reset_index()
daily["share"] = daily["wind_solar"] / daily["load"]
daily["date_local"] = pd.to_datetime(daily["date_local"])
daily["doy"] = daily["date_local"].dt.dayofyear

calendar = pd.DataFrame(index=range(2019,2027), columns=range(1,367), dtype=float)
for row in daily.itertuples():
    calendar.loc[row.year, row.doy] = row.share

fig, ax = plt.subplots(figsize=(14, 5.8))
im = ax.imshow(calendar.values, aspect="auto", vmin=0, vmax=1, cmap=WIND_SOLAR_CMAP)
ax.set_title("How does wind-and-solar support vary through the year?", loc="left")
ax.set_yticks(range(8), [str(y) if y < 2026 else "2026 · partial" for y in calendar.index])
ax.set_xticks([0,49,99,149,199,249,299,349], [1,50,100,150,200,250,300,350])
ax.set_xlabel("Day of calendar year")
cb = fig.colorbar(im, ax=ax, fraction=.025, pad=.02)
cb.set_label("Wind + solar / load")
plt.tight_layout()
plt.show()

**Interpretation.** The calendar view shows long episodes of stronger and weaker wind+solar support as well as the seasonal solar contribution. It is useful for spotting persistence that can disappear in monthly averages.

## 18. Which variables co-move?

Spearman correlation is a simple starting point for association. Residual load is derived from load, wind and solar, so correlations involving those shared components are descriptive rather than independent evidence.

In [ ]:
corr_cols = ["Grid Load", "Wind Onshore", "Wind Offshore", "Solar", "Residual Load", "Wind+solar / load"]
corr = valid[corr_cols].corr(method="spearman")
labels = ["Grid load", "Onshore wind", "Offshore wind", "Solar", "Residual load", "Wind+solar share"]

fig, ax = plt.subplots(figsize=(8.2, 7))
im = ax.imshow(corr.values, vmin=-1, vmax=1, cmap=DIV_CMAP)
ax.set_title("Which system variables tend to move together?", loc="left")
ax.set_xticks(range(len(labels)), labels, rotation=25, ha="right")
ax.set_yticks(range(len(labels)), labels)
for i in range(len(labels)):
    for j in range(len(labels)):
        value = corr.iloc[i, j]
        text_color = "white" if abs(value) >= 0.58 else COLORS["ink"]
        ax.text(
            j, i, f"{value:.2f}",
            ha="center", va="center",
            fontsize=9.2,
            color=text_color,
            fontweight="semibold" if abs(value) >= 0.58 else "normal",
        )
cb = fig.colorbar(im, ax=ax, fraction=.045, pad=.04)
cb.set_label("Spearman r")
plt.tight_layout()
plt.show()

**Interpretation.** Wind and solar support is strongly related to lower residual load by construction, while onshore and offshore wind also co-move substantially. Correlation is useful for orientation, but not for causal conclusions or feature selection on its own.

## 19. Main takeaways

- The SMARD record is sufficiently complete to support a long 2019–2026 descriptive analysis, with 2026 explicitly partial.
- Residual load has a strong daily and seasonal geometry rather than behaving like a stationary level-only series.
- Harmonics, ACF and PACF show recurring structure that can motivate later time-aware features, but they do not by themselves prove prediction-time availability.
- Short-term ramp behaviour has its own tail and seasonal/hourly structure, so level and change should be analysed separately.
- Wind+solar support is persistent over multi-day periods and is mechanically related to residual load through the accounting identity.